# Project 1
# Amna Malik

In [1]:
from datetime import datetime
now = datetime.now()
print(f"Last updated: {now.month}/{now.day}/{now.year} {now.strftime('%I:%M%p').lstrip('0').lower()}")

Last updated: 2/13/2026 10:36pm


In [2]:
import pandas as pd
import numpy as np

# load datasets
df_reg = pd.read_csv("MRegularSeasonDetailedResults.csv")
df_tourney = pd.read_csv("MNCAATourneyDetailedResults.csv")

# add tournament indicator
df_reg["Is_Tourney"] = 0
df_tourney["Is_Tourney"] = 1

# combine games
df_games = pd.concat([df_reg, df_tourney], ignore_index=True)

# keep required seasons
df_games = df_games[(df_games["Season"] >= 2003) & (df_games["Season"] <= 2025)].copy()

# equation 1 - possessions
df_games["W_Poss"] = df_games["WFGA"] - df_games["WOR"] + df_games["WTO"] + 0.475 * df_games["WFTA"]
df_games["L_Poss"] = df_games["LFGA"] - df_games["LOR"] + df_games["LTO"] + 0.475 * df_games["LFTA"]

# equation 2 - effective field goal percentage (eFG)
df_games["W_EFG"] = (df_games["WFGM"] + 0.5 * df_games["WFGM3"]) / df_games["WFGA"]
df_games["L_EFG"] = (df_games["LFGM"] + 0.5 * df_games["LFGM3"]) / df_games["LFGA"]

# equation 3 - turnover rate (TOV_rate)
df_games["W_TOV_rate"] = (df_games["WTO"] / df_games["W_Poss"]) * 100
df_games["L_TOV_rate"] = (df_games["LTO"] / df_games["L_Poss"]) * 100

# equation 4 - offensive rebound rate (ORB_rate)
df_games["W_ORB_rate"] = df_games["WOR"] / (df_games["WOR"] + df_games["LDR"])
df_games["L_ORB_rate"] = df_games["LOR"] / (df_games["LOR"] + df_games["WDR"])

# equation 5 - free throw rate (FTR_rate)
df_games["W_FTR_rate"] = df_games["WFTA"] / df_games["WFGA"]
df_games["L_FTR_rate"] = df_games["LFTA"] / df_games["LFGA"]

# randomize team A and team B
np.random.seed(42)
swap = np.random.rand(len(df_games)) < 0.5

df_games["TeamA_ID"] = np.where(swap, df_games["WTeamID"], df_games["LTeamID"])
df_games["TeamB_ID"] = np.where(swap, df_games["LTeamID"], df_games["WTeamID"])

# target variable
df_games["Win_A"] = np.where(df_games["TeamA_ID"] == df_games["WTeamID"], 1, 0)

# differential features
A_EFG = np.where(swap, df_games["W_EFG"], df_games["L_EFG"])
B_EFG = np.where(swap, df_games["L_EFG"], df_games["W_EFG"])

A_TOV = np.where(swap, df_games["W_TOV_rate"], df_games["L_TOV_rate"])
B_TOV = np.where(swap, df_games["L_TOV_rate"], df_games["W_TOV_rate"])

A_ORB = np.where(swap, df_games["W_ORB_rate"], df_games["L_ORB_rate"])
B_ORB = np.where(swap, df_games["L_ORB_rate"], df_games["W_ORB_rate"])

A_FTR = np.where(swap, df_games["W_FTR_rate"], df_games["L_FTR_rate"])
B_FTR = np.where(swap, df_games["L_FTR_rate"], df_games["W_FTR_rate"])

df_games["DIFF_EFG"] = A_EFG - B_EFG
df_games["DIFF_TOV"] = A_TOV - B_TOV
df_games["DIFF_ORB"] = A_ORB - B_ORB
df_games["DIFF_FTR"] = A_FTR - B_FTR

# create master analytical table
df_mat = df_games[[
    "Season",
    "Is_Tourney",
    "TeamA_ID",
    "TeamB_ID",
    "DIFF_EFG",
    "DIFF_TOV",
    "DIFF_ORB",
    "DIFF_FTR",
    "Win_A"
]].copy()

df_mat = df_mat.dropna().drop_duplicates()

# export csv
df_mat.to_csv("Master_Analytical_Table.csv", index=False)

# sanity checks
df_mat.head()
df_mat[["DIFF_EFG", "Win_A"]].corr()


,DIFF_EFG,Win_A
DIFF_EFG,1.000000,0.660288
Win_A,0.660288,1.000000
